# Level 1 Source Audit — EODH and EODI

This first Level 1 step updates the source catalog and documents the source contract. It reads file metadata, workbook sheet dimensions, field headers, and the EODI workbook's `DICC` documentation sheet. It does not read survey observation rows, calculate weights, join sources, or construct OD matrices.

EODH remains the primary source for the observed household travel matrix. EODI is a separate intercept-survey operation. Stage V is methodological context for a future, separately reported hybrid product; it does not authorize adding EODI values to or replacing EODH results.

The blank first header in the EODI `BD` sheet is retained as blank. `ID` appears in the source dictionary as a candidate interpretation, but it is not assigned to that header until a later empirical validation.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'data/sources/raw').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the notebook path.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import pandas as pd
from mode_choice.source_inventory import build_inventory, write_catalogs
from mode_choice.level1_source_audit import build_eodi_workbook_catalog, write_eodi_workbook_catalogs

print(f'Project root: {PROJECT_ROOT}')

Project root: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect


## 1. Regenerate the full source inventory

The inventory includes EODH, EODI, survey documentation, contract reports, and spatial reference files. CSV scanning reads only headers; Excel and PDF scanning reads structural metadata. No source files are changed.

In [2]:
inventory, schemas = build_inventory(PROJECT_ROOT)
catalog_path, schema_path = write_catalogs(PROJECT_ROOT)
catalog = pd.DataFrame(inventory)
print(f'Cataloged {len(catalog)} files.')
print(f'Inventory: {catalog_path.relative_to(PROJECT_ROOT)}')
print(f'CSV schema: {schema_path.relative_to(PROJECT_ROOT)}')
display(catalog.groupby('category', dropna=False).agg(files=('source_path', 'count'), total_size_bytes=('size_bytes', 'sum')).sort_index())
display(catalog.loc[catalog['source_path'].str.contains('/EODI/', case=False, regex=False), ['source_path', 'category', 'format', 'size_bytes', 'metadata_status']].sort_values('source_path'))

Cataloged 80 files.
Inventory: docs\data_dictionary\source_inventory.csv
CSV schema: docs\data_dictionary\source_schema.json


,files,total_size_bytes
category,,
contract_context,18,51147649
em2023_documentation,5,57015665
em2023_microdata,5,165077421
eodi_counts_and_geospatial_references,23,1328753
eodi_intercept_survey_data,1,37994539
eodi_survey_instruments,3,823903
eodi_technical_documentation,3,1752699
geographic_reference,16,21415448
survey_instruments,3,1515309


,source_path,category,format,size_bytes,metadata_status
31,data/sources/raw/EODI/01_Datos tecnicos de la ...,eodi_technical_documentation,pdf,518852,ok
32,data/sources/raw/EODI/01_Datos tecnicos de la ...,eodi_technical_documentation,xlsx,649667,ok
33,data/sources/raw/EODI/01_Datos tecnicos de la ...,eodi_technical_documentation,xlsx,584180,ok
34,data/sources/raw/EODI/02_Formularios/a. Transp...,eodi_survey_instruments,pdf,267416,ok
35,data/sources/raw/EODI/02_Formularios/b. Transp...,eodi_survey_instruments,pdf,288150,ok
36,data/sources/raw/EODI/02_Formularios/c. Transp...,eodi_survey_instruments,pdf,268337,ok
37,data/sources/raw/EODI/03_Aforos/a. Base de dat...,eodi_counts_and_geospatial_references,xlsx,704657,ok
38,data/sources/raw/EODI/03_Aforos/b. Estaciones_...,eodi_counts_and_geospatial_references,cpg,5,file_metadata_only
39,data/sources/raw/EODI/03_Aforos/b. Estaciones_...,eodi_counts_and_geospatial_references,dbf,241974,file_metadata_only
40,data/sources/raw/EODI/03_Aforos/b. Estaciones_...,eodi_counts_and_geospatial_references,prj,425,file_metadata_only


## 2. Catalog EODI workbook structure and documented fields

The detailed EODI catalog reads sheet names and dimensions, the `BD` header row, and dictionary text from `DICC`. Observation rows are not read. The dictionary definitions remain in Spanish where they are quoted from the source; the source wording is preserved rather than translated or reinterpreted.

In [3]:
workbook_rows, field_rows = build_eodi_workbook_catalog(PROJECT_ROOT)
workbooks = pd.DataFrame(workbook_rows)
fields = pd.DataFrame(field_rows)
workbook_catalog_path, field_dictionary_path = write_eodi_workbook_catalogs(PROJECT_ROOT)
print(f'Workbook sheets cataloged: {len(workbooks)}')
print(f'Field/header metadata rows: {len(fields)}')
print(f'Workbook catalog: {workbook_catalog_path.relative_to(PROJECT_ROOT)}')
print(f'Field dictionary: {field_dictionary_path.relative_to(PROJECT_ROOT)}')
display(workbooks)
display(fields.loc[fields['sheet_name'].eq('BD'), ['source_role', 'source_position_1based', 'source_field_name', 'header_status']])
display(fields.loc[fields['sheet_name'].eq('DICC'), ['source_field_name', 'documented_definition', 'documented_values_or_range', 'definition_status']])

Workbook sheets cataloged: 8
Field/header metadata rows: 80
Workbook catalog: docs\data_dictionary\eodi_workbook_catalog.csv
Field dictionary: docs\data_dictionary\eodi_field_dictionary.csv


,source_role,source_path,sheet_name,max_row_including_header,max_column,observation_rows_if_header_row
0,intercept_survey,data/sources/raw/EODI/04_Base datos/a. Base de...,PORTADA,64,15,63
1,intercept_survey,data/sources/raw/EODI/04_Base datos/a. Base de...,CONTENIDO,32,13,31
2,intercept_survey,data/sources/raw/EODI/04_Base datos/a. Base de...,DICC,34,4,33
3,intercept_survey,data/sources/raw/EODI/04_Base datos/a. Base de...,BD,200946,33,200945
4,traffic_counts,data/sources/raw/EODI/03_Aforos/a. Base de dat...,PORTADA,64,15,63
5,traffic_counts,data/sources/raw/EODI/03_Aforos/a. Base de dat...,CONTENIDO,32,13,31
6,traffic_counts,data/sources/raw/EODI/03_Aforos/a. Base de dat...,DICC,8,4,7
7,traffic_counts,data/sources/raw/EODI/03_Aforos/a. Base de dat...,BD,3957,7,3956


,source_role,source_position_1based,source_field_name,header_status
33,intercept_survey,1,,blank_source_header
34,intercept_survey,2,Longitud_estación,source_header
35,intercept_survey,3,Latitud_estación,source_header
36,intercept_survey,4,ID_estación,source_header
37,intercept_survey,5,Tipo,source_header
38,intercept_survey,6,Periodo,source_header
39,intercept_survey,7,Sentido,source_header
40,intercept_survey,8,Categoría,source_header
41,intercept_survey,9,tipo_vehiculo/servicio,source_header
42,intercept_survey,10,ocupacion,source_header


,source_field_name,documented_definition,documented_values_or_range,definition_status
0,ID,Código único de encuesta,Código único que relaciona el número de la enc...,source_dictionary_text; field_match_pending
1,Longitud_estación,Longitud del punto de encuesta,Número con decimales que indica la longitud de...,source_dictionary_text; field_match_pending
2,Latitud_estación,Latitud del punto de encuesta,Número con decimales que indica la latitud del...,source_dictionary_text; field_match_pending
3,ID_estación,Identificador de la estación donde se realizo ...,"Estaciones de transporte privado, Bicicleta en...",source_dictionary_text; field_match_pending
4,Tipo,Clase de encuesta por agrupación general del m...,T1: Transporte Privado\nT2: Bicicleta en ciclo...,source_dictionary_text; field_match_pending
5,Periodo,Franja horaria de realización de la encuesta,AM: encuestas aplicadas en el período de la ma...,source_dictionary_text; field_match_pending
6,Sentido,Dirección de circulación del tráfico,Sentido cardinal en que circulaba el encuestad...,source_dictionary_text; field_match_pending
7,Categoría,Se refiere a la tipología vehicular encuestada,Auto\nMoto\nBicicleta en calzada\nBicicleta en...,source_dictionary_text; field_match_pending
8,tipo_vehiculo/servicio,Unidad de transporte en el que se movilizaba l...,Auto\nMoto\nBicicleta Convencional\nBicicleta ...,source_dictionary_text; field_match_pending
9,ocupacion,Ocupación del vehículo sin incluir al condutor...,"De acuerdo al tipo de vehículo encuestado, se ...",source_dictionary_text; field_match_pending


## 3. Source contract and review gate

- **EODH:** primary source for a sample-based observed household travel OD product. Eligibility, geography, and expansion-factor domains remain subject to documentation and validation.
- **EODI:** separate intercept-survey operation with station, period, direction, and mode-related dimensions. Its factor and double-count treatment are not yet validated.
- **Traffic counts:** peak-hour volumes are count data. Their unit and coverage must be aligned with the EODI survey dimensions before any expansion diagnostic; they are not an EODH weight.
- **Spatial files:** inventory records the files and formats only. Zone coverage, identifiers, coordinate reference systems, and joins are not validated here.
- **Stage V:** informs a future separate hybrid reference matrix, using EODI where applicable and EODH for OD pairs absent from EODI, subject to compatible periods, modes, geography, and definitions. It is not an automatic pooling or calibration rule.

Review the generated source inventory and EODI field catalog before proceeding. The next dependent step is to define and validate the unweighted EODH trip eligibility and OD geography rules. No matrix or weight calculations are part of this notebook.